# Overture Maps Places — 49-site food and shopping context features

## What this data source is

[Overture Maps Places](https://docs.overturemaps.org/guides/places/) is an open global point-of-interest dataset assembled from providers including Meta, Microsoft and Foursquare. A Place represents a named, publicly observable destination such as a restaurant, shop, service, hospital or landmark. The project pins release **2026-08-19.0** so all 49 sites are measured against one source snapshot.

### API and access method

The notebook uses the official [`overturemaps` Python client](https://github.com/OvertureMaps/overturemaps-py). No API key or account is required. The client discovers and reads Overture's public cloud-hosted GeoParquet partitions, and the module requests only a conservative bounding box around each site. Each first-run response is cached locally as GeoParquet under `data/external_raw/overture_places/2026-08-19.0/`; this cache is excluded from Git. Exact 500 m geodesic filtering is applied after download, so records in the corners of the query rectangle do not enter the features.

## What information can be obtained

A Place may provide a GERS identifier, point geometry, name, `basic_category`, a detailed `taxonomy` hierarchy, confidence, operating status and source attribution. The official hierarchy is designed for aggregation: a specialised restaurant can be counted through the `food_and_drink` ancestor, while a grocery store can be counted through its `shopping > food_and_beverage_store` path. The deprecated `categories` property is not used.

## Features produced by this notebook

| Output field | Source-based definition |
|---|---|
| `overture_food_drink_count_500m` | Places within 500 m whose official taxonomy hierarchy contains `food_and_drink`; this includes restaurants, cafes, bars and other descendants of that official top-level category. |
| `overture_grocery_convenience_pharmacy_count_500m` | Places within 500 m whose hierarchy contains `food_and_beverage_store`, `convenience_store` or `pharmacy_and_drug_store`. The explicit name avoids presenting the project grouping as an official Overture category. |
| `overture_shopping_mall_present_500m` | Whether at least one eligible Place within 500 m has `shopping_mall` in its taxonomy hierarchy. Overture does not provide reliable store floor area, so this field does not infer that an ordinary shop is a major destination. |
| `overture_other_shopping_count_500m` | Other Places under the official `shopping` hierarchy after excluding the grocery/convenience/pharmacy group and shopping malls. |
| `overture_places_data_status` | Source-processing status. It distinguishes a completed calculation with at least one target place from a completed calculation with none. It is a QA field, not a pedestrian-flow predictor. |

## Scientific scope and source limitations

The 500 m radius is the same straight-line point catchment used by the other approved context sources; it is not an SLK road segment or a walking-network service area. Explicit `temporarily_closed` and `permanently_closed` records are excluded. A missing `operating_status` is included but is not relabelled as confirmed open: Overture defines status independently of opening hours, and the current WA records often leave it null. No additional confidence threshold is imposed because the project has not validated a defensible local cutoff; confidence is retained in source QA rather than added as another model feature. Counts remain subject to source coverage, stale records and taxonomy error.

Overture Places uses source-specific CDLA Permissive 2.0 and Apache 2.0 licensing. The notebook records the release and source method, and the project should retain the attribution guidance at [Overture attribution and licensing](https://docs.overturemaps.org/attribution/).

## Reusable implementation

Reusable acquisition, caching, cleaning, geodesic filtering, taxonomy aggregation, QA and safe provenance live in `src/pedflow/context/sources/overture_places.py`. The notebook remains the scientific explanation and review surface. The module accepts the already validated `sites` DataFrame; it does not reopen `model_dataset.xlsx`, write the integrated model table or contain notebook display code.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from openpyxl.styles import Alignment, Font, PatternFill


def find_project_root() -> Path:
    """Find the repository root from any supported working directory."""
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src" / "pedflow").is_dir():
            return path
    raise FileNotFoundError(
        f"Project root not found above {current}. Expected pyproject.toml and src/pedflow/."
    )


PROJECT_ROOT = find_project_root()
SOURCE_PACKAGE = PROJECT_ROOT / "src"
if str(SOURCE_PACKAGE) not in sys.path:
    sys.path.insert(0, str(SOURCE_PACKAGE))

from pedflow.context.contracts import SourceConfig
from pedflow.context.sources.overture_places import FEATURE_COLUMNS, extract
from pedflow.context.validation import (
    validate_and_normalise_sites,
    validate_source_features,
)


MODEL_DATASET_FILE = PROJECT_ROOT / "data" / "processed" / "model_dataset.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "02_feature_extraction"
FEATURE_OUTPUT_FILE = OUTPUT_DIR / "02_09_haoqi_overture_places_features.xlsx"
SOURCE_CONFIG = SourceConfig(
    name="overture_places",
    enabled=True,
    options={
        "release": "2026-08-19.0",
        "buffer_radius_m": 500,
        "cache_directory": "data/external_raw/overture_places/{release}",
        "allow_download": True,
        "connect_timeout_seconds": 15,
        "request_timeout_seconds": 120,
        "max_workers": 4,
    },
)

if not MODEL_DATASET_FILE.is_file():
    raise FileNotFoundError(
        f"Model dataset not found: {MODEL_DATASET_FILE}\n"
        "Expected the authoritative workbook at data/processed/model_dataset.xlsx"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root  :", PROJECT_ROOT)
print("Model dataset :", MODEL_DATASET_FILE.relative_to(PROJECT_ROOT))
print("Feature output:", FEATURE_OUTPUT_FILE.relative_to(PROJECT_ROOT))

Project root  : /workspace
Model dataset : data/processed/model_dataset.xlsx
Feature output: 06_outputs/02_feature_extraction/02_09_haoqi_overture_places_features.xlsx


## 1. Read the authoritative model sites and run the bounded source acquisition

`Model_Data` remains the authoritative site table. Only `site_id`, latitude, longitude and the declared WGS84 CRS are passed to the source module. Targets, road attributes, imagery fields and other context features are not copied into this source-specific output. The extractor has no fixed row-count assumption, although the current reviewed input contains 49 sites.

For each site, the module constructs the smallest cardinal WGS84 bounding box that contains a 500 m geodesic circle. The official client retrieves candidate Place points in that box; the deterministic calculation then measures exact geodesic distance from the site and removes corner records outside 500 m.

In [2]:
model_data = pd.read_excel(
    MODEL_DATASET_FILE,
    sheet_name="Model_Data",
    dtype={"site_id": "string", "coordinate_crs": "string"},
)
sites = validate_and_normalise_sites(model_data)

result = extract(sites, SOURCE_CONFIG, PROJECT_ROOT)
overture_features = validate_source_features(
    source_name=result.source_name,
    features=result.features,
    expected_site_ids=sites["site_id"],
    column_prefixes=("overture_",),
)

print(f"Sites supplied     : {len(sites)}")
print(f"Source release     : {result.source_version}")
print(f"Cache hits         : {result.metadata['cache_hit_count']}")
print(f"New source downloads: {result.metadata['download_count']}")

Sites supplied     : 49
Source release     : 2026-08-19.0
Cache hits         : 49
New source downloads: 0


## 2. Inspect source provenance and coverage before interpreting counts

The rectangular source-record count is expected to exceed the exact-buffer count. `within_buffer_count` is the relevant spatial denominator for source QA. `eligible_record_count` removes only records explicitly marked temporarily or permanently closed; it does not treat a null status as confirmed open. `missing_taxonomy_count` identifies records that exist spatially but cannot enter the four category aggregates.

The cache-set SHA-256 binds this run to the exact local per-site snapshots. It is provenance, not a model input. The raw cache contains source records and remains outside Git; the final workbook contains only aggregate site-level features.

In [3]:
source_run_summary = pd.Series(
    {
        "source_version": result.source_version,
        "retrieval_method": result.metadata["retrieval_method"],
        "buffer_radius_m": result.metadata["buffer_radius_m"],
        "cache_directory": result.metadata["cache_directory"],
        "cache_hit_count": result.metadata["cache_hit_count"],
        "download_count": result.metadata["download_count"],
        "rectangular_source_records": result.metadata["source_record_count"],
        "records_within_exact_500m": result.metadata["within_buffer_count"],
        "eligible_records": result.metadata["eligible_record_count"],
        "explicit_closed_records_excluded": result.metadata["excluded_closed_count"],
        "operating_status_counts": result.metadata["operating_status_counts"],
        "cache_set_sha256": result.metadata["cache_set_sha256"],
    },
    name="value",
).to_frame()

display(source_run_summary)
if result.warnings:
    print("Source warnings:")
    for warning in result.warnings:
        print("-", warning)
else:
    print("Source warnings: none")

,value
source_version,2026-08-19.0
retrieval_method,"Official overturemaps Python client, bounded p..."
buffer_radius_m,500.0
cache_directory,data/external_raw/overture_places/2026-08-19.0
cache_hit_count,49
download_count,0
rectangular_source_records,3488
records_within_exact_500m,3075
eligible_records,3075
explicit_closed_records_excluded,0


Source warnings:
- 3075 in-buffer records have no operating_status; they are included because the source does not mark them closed.
- 106 in-buffer records have no taxonomy hierarchy and cannot enter the four category aggregates.
- 2 site(s) have no eligible food/drink or shopping place in 500 m.


## 3. Review the extracted feature situation across all supplied sites

The four environmental measures are mutually interpretable but not guaranteed statistically independent. Food/drink uses the official top-level ancestor. The three shopping outputs partition the target shopping hierarchy into routine food/convenience/pharmacy destinations, shopping-mall presence, and remaining shopping places. A shopping mall is represented as presence because one mall record can contain many internal businesses and should not be treated as equivalent to one small shop.

The displayed table is the actual source-specific result. The status distribution and numeric summary show whether zeros are concentrated, whether a few commercial centres dominate the scale, and whether later transformations such as `log1p` may be considered inside model cross-validation. Feature selection and transformation are not performed here.

In [4]:
features = overture_features.loc[:, list(FEATURE_COLUMNS)].copy()

display(features)
display(
    features["overture_places_data_status"]
    .value_counts(dropna=False)
    .to_frame("sites")
)
display(
    features.drop(
        columns=["site_id", "overture_places_data_status"]
    ).describe(include="all").T
)

,site_id,overture_places_data_status,overture_food_drink_count_500m,overture_grocery_convenience_pharmacy_count_500m,overture_shopping_mall_present_500m,overture_other_shopping_count_500m
0,250303-01,complete_no_eligible_places,0,0,False,0
1,250303-02,complete,3,0,False,19
2,250303-03,complete,19,8,True,35
3,250303-04,complete,14,13,False,20
4,250303-05,complete,20,7,True,19
5,250303-06,complete_no_eligible_places,0,0,False,0
6,250303-07,complete,14,5,False,11
7,250303-08,complete,5,4,False,6
8,250303-09,complete,5,5,False,8
9,250303-10,complete,14,8,False,30


,sites
overture_places_data_status,
complete,47
complete_no_eligible_places,2


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
overture_food_drink_count_500m,49.0,NaN,NaN,NaN,6.346939,6.811605,0.0,2.0,3.0,8.0,25.0
overture_grocery_convenience_pharmacy_count_500m,49.0,NaN,NaN,NaN,3.020408,2.918974,0.0,0.0,3.0,5.0,13.0
overture_shopping_mall_present_500m,49,2,False,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
overture_other_shopping_count_500m,49.0,NaN,NaN,NaN,8.510204,8.321965,0.0,2.0,6.0,12.0,35.0


## 4. Save one independent, single-sheet feature table

The workbook is not a copy of `model_dataset.xlsx`. It contains one row per supplied `site_id`, the source QA status and the four approved Overture fields only. Its single worksheet is `Overture_Places_Features`, so ordinary `pd.read_excel(path)` use remains unambiguous. Later modelling code can merge it on `site_id` using `how="left"` and `validate="one_to_one"`.

The output name follows the notebook stem: `02_09_haoqi_overture_places_features.xlsx`, stored directly in `06_outputs/02_feature_extraction/`. A round-trip assertion verifies that the saved workbook preserves identifiers, statuses, counts and booleans.

In [5]:
with pd.ExcelWriter(FEATURE_OUTPUT_FILE, engine="openpyxl") as writer:
    features.to_excel(
        writer,
        sheet_name="Overture_Places_Features",
        index=False,
    )
    worksheet = writer.sheets["Overture_Places_Features"]
    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions
    worksheet.sheet_view.showGridLines = False

    header_fill = PatternFill("solid", fgColor="1F4E78")
    for cell in worksheet[1]:
        cell.fill = header_fill
        cell.font = Font(color="FFFFFF", bold=True)
        cell.alignment = Alignment(
            horizontal="center", vertical="center", wrap_text=True
        )
    worksheet.row_dimensions[1].height = 48

    for column, width in {
        "A": 14,
        "B": 38,
        "C": 34,
        "D": 51,
        "E": 43,
        "F": 39,
    }.items():
        worksheet.column_dimensions[column].width = width

saved_features = pd.read_excel(
    FEATURE_OUTPUT_FILE,
    sheet_name="Overture_Places_Features",
    dtype={"site_id": "string", "overture_places_data_status": "string"},
)
pd.testing.assert_frame_equal(
    saved_features,
    features,
    check_dtype=False,
    check_exact=True,
)

display(saved_features)
print("Saved:", FEATURE_OUTPUT_FILE.relative_to(PROJECT_ROOT))

,site_id,overture_places_data_status,overture_food_drink_count_500m,overture_grocery_convenience_pharmacy_count_500m,overture_shopping_mall_present_500m,overture_other_shopping_count_500m
0,250303-01,complete_no_eligible_places,0,0,False,0
1,250303-02,complete,3,0,False,19
2,250303-03,complete,19,8,True,35
3,250303-04,complete,14,13,False,20
4,250303-05,complete,20,7,True,19
5,250303-06,complete_no_eligible_places,0,0,False,0
6,250303-07,complete,14,5,False,11
7,250303-08,complete,5,4,False,6
8,250303-09,complete,5,5,False,8
9,250303-10,complete,14,8,False,30


Saved: 06_outputs/02_feature_extraction/02_09_haoqi_overture_places_features.xlsx
